In [ ]:
import pandas as pd
from pathlib import Path

# Load the original datasets

orders = pd.read_csv("/content/order-items.csv")
products = pd.read_csv("/content/products.csv")

# Check dataset sizes
print("Order items shape:", orders.shape)
print("Products shape:", products.shape)

# Inspect column names and data types
print("\nORDER ITEMS")
orders.info()

print("\nPRODUCTS")
products.info()

# Preview the first five rows
display(orders.head())
display(products.head())

Order items shape: (57373, 9)
Products shape: (32341, 9)

ORDER ITEMS
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 57373 entries, 0 to 57372
Data columns (total 9 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   order_id               57373 non-null  object 
 1   order_item_id          57373 non-null  int64  
 2   product_id             57373 non-null  object 
 3   seller_id              57373 non-null  object 
 4   shipping_limit_date    57373 non-null  object 
 5   price                  57373 non-null  float64
 6   freight_value          57373 non-null  float64
 7   product_category_name  15946 non-null  object 
 8   product_weight         15946 non-null  float64
dtypes: float64(3), int64(1), object(5)
memory usage: 3.9+ MB

PRODUCTS
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32341 entries, 0 to 32340
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------  

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,product_category_name,product_weight
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,9/19/2017 9:45,58.90,13.29,cool_stuff,650.0
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,5/3/2017 11:05,239.90,19.93,pet_shop,30000.0
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,1/18/2018 14:48,199.00,17.87,moveis_decoracao,3050.0
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,8/15/2018 10:10,12.99,12.79,perfumaria,200.0
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2/13/2017 13:57,199.90,18.14,ferramentas_jardim,3750.0


,product_id,product_category_name,product_name_length,product_description_length,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40,287,1,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44,276,1,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46,250,1,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27,261,1,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37,402,4,625.0,20.0,17.0,13.0


In [ ]:
# Missing values
print("ORDER ITEMS: Missing values")
display(orders.isna().sum().sort_values(ascending=False))

print("PRODUCTS: Missing values")
display(products.isna().sum().sort_values(ascending=False))

ORDER ITEMS: Missing values


,0
product_category_name,41427
product_weight,41427
order_id,0
order_item_id,0
product_id,0
shipping_limit_date,0
seller_id,0
freight_value,0
price,0


PRODUCTS: Missing values


,0
product_width_cm,1
product_length_cm,1
product_height_cm,1
product_weight_g,1
product_id,0
product_photos_qty,0
product_description_length,0
product_name_length,0
product_category_name,0


In [ ]:
# Completely duplicated rows
print("Duplicate rows in order items:", orders.duplicated().sum())
print("Duplicate rows in products:", products.duplicated().sum())

Duplicate rows in order items: 0
Duplicate rows in products: 0


In [ ]:
# Check key columns
print(
    "Duplicate order IDs:",
    orders["order_id"].duplicated().sum()
)

print(
    "Duplicate product IDs in products:",
    products["product_id"].duplicated().sum()
)

Duplicate order IDs: 0
Duplicate product IDs in products: 0


In [ ]:
# Check missing product matches
unmatched = ~orders["product_id"].isin(products["product_id"])

print("Unmatched product records:", unmatched.sum())
print("Unmatched percentage:", round(unmatched.mean() * 100, 2), "%")

Unmatched product records: 820
Unmatched percentage: 1.43 %


In [ ]:
orders["shipping_limit_date"] = pd.to_datetime(
    orders["shipping_limit_date"],
    errors="coerce"
)

print("Invalid dates:", orders["shipping_limit_date"].isna().sum())
print("Earliest date:", orders["shipping_limit_date"].min())
print("Latest date:", orders["shipping_limit_date"].max())

Invalid dates: 0
Earliest date: 2016-10-08 13:46:00
Latest date: 2020-02-05 03:30:00


In [ ]:
# Keep the original datasets unchanged
orders_work = orders.copy()
products_work = products.copy()

# Rename existing columns to avoid confusion after joining
orders_work = orders_work.rename(columns={
    "product_category_name": "category_from_order",
    "product_weight": "weight_from_order_g"
})

products_work = products_work.rename(columns={
    "product_category_name": "category_from_product",
    "product_weight_g": "weight_from_product_g"
})

# Join using product_id
merged = orders_work.merge(
    products_work,
    on="product_id",
    how="left",
    validate="many_to_one",
    indicator=True
)

# Prefer product master information, using existing values as fallback
merged["product_category_name"] = (
    merged["category_from_product"]
    .combine_first(merged["category_from_order"])
)

merged["product_weight_g"] = (
    merged["weight_from_product_g"]
    .combine_first(merged["weight_from_order_g"])
)

# Record whether each product ID matched
merged["product_match_status"] = merged["_merge"].map({
    "both": "Matched",
    "left_only": "Unmatched"
})

# Remove temporary columns
merged = merged.drop(columns=[
    "category_from_order",
    "weight_from_order_g",
    "category_from_product",
    "weight_from_product_g",
    "_merge"
])

print("Original order-item rows:", len(orders))
print("Rows after joining:", len(merged))

print("\nProduct match status:")
print(merged["product_match_status"].value_counts())

print("\nMissing values after joining:")
print(
    merged[[
        "product_category_name",
        "product_weight_g"
    ]].isna().sum()
)

display(merged.head())

Original order-item rows: 57373
Rows after joining: 57373

Product match status:
product_match_status
Matched      56553
Unmatched      820
Name: count, dtype: int64

Missing values after joining:
product_category_name    820
product_weight_g         820
dtype: int64


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,product_name_length,product_description_length,product_photos_qty,product_length_cm,product_height_cm,product_width_cm,product_category_name,product_weight_g,product_match_status
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:00,58.90,13.29,58.0,598.0,4.0,28.0,9.0,14.0,cool_stuff,650.0,Matched
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:00,239.90,19.93,56.0,239.0,2.0,50.0,30.0,40.0,pet_shop,30000.0,Matched
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:00,199.00,17.87,59.0,695.0,2.0,33.0,13.0,33.0,moveis_decoracao,3050.0,Matched
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:00,12.99,12.79,42.0,480.0,1.0,16.0,10.0,15.0,perfumaria,200.0,Matched
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:00,199.90,18.14,59.0,409.0,1.0,35.0,40.0,30.0,ferramentas_jardim,3750.0,Matched


In [ ]:
print("Missing categories:", merged["product_category_name"].isna().sum())
print("Missing weights:", merged["product_weight_g"].isna().sum())

print(
    "Rows with zero freight:",
    (merged["freight_value"] == 0).sum()
)

print(
    "Rows with negative freight:",
    (merged["freight_value"] < 0).sum()
)

print(
    "Rows with item ID greater than 1:",
    (merged["order_item_id"] > 1).sum()
)

print(
    "Unique order IDs:",
    merged["order_id"].nunique()
)

Missing categories: 820
Missing weights: 820
Rows with zero freight: 188
Rows with negative freight: 0
Rows with item ID greater than 1: 7182
Unique order IDs: 57373


In [ ]:
# Check order and item structure
print("Unique order IDs:", orders_work["order_id"].nunique())

print("\nOrder item ID distribution:")
print(orders_work["order_item_id"].value_counts().sort_index())

print(
    "\nOrders containing multiple distinct item IDs:",
    (orders_work.groupby("order_id")["order_item_id"].nunique() > 1).sum()
)

# Inspect examples where item ID is greater than 1
display(
    orders_work.loc[
        orders_work["order_item_id"] > 1,
        ["order_id", "order_item_id", "product_id", "price", "freight_value"]
    ].head(10)
)

Unique order IDs: 57373

Order item ID distribution:
order_item_id
1     50191
2      5009
3      1169
4       505
5       247
6       138
7        27
8        18
9        13
10       13
11       12
12        8
13        5
14        4
15        3
16        2
17        2
18        2
19        2
20        2
21        1
Name: count, dtype: int64

Orders containing multiple distinct item IDs: 0


,order_id,order_item_id,product_id,price,freight_value
14,0009792311464db532ff765bf7b182ae,2,368c6c730842d78016ad823897a372db,49.90,13.37
33,0015ebb40fb17286bea51d4607c4733c,2,e95ee6822b66ac6058e2e4aff656071a,21.33,15.10
34,00169e31ef4b29deaae414f9a5e95929,3,e95ee6822b66ac6058e2e4aff656071a,21.33,15.10
43,001d8f0e34a38c37f7dba2a37d4eba8b,2,0b0172eb0fd18479d29c3bc122c058c2,24.89,17.63
44,001daeb0eddc45b999bad0801ad9d273,3,0b0172eb0fd18479d29c3bc122c058c2,24.89,17.63
49,0020a222f55eb79a372d0efee3cca688,2,e67307ff0f15ade43fcb6e670be7a74c,18.99,7.78
76,00310b0c75bb13015ec4d82d341865a4,2,2d9ff06c8870a518f5f6909774e140fb,78.00,8.90
81,00335f75ea6a4455b524a8dd177067f5,2,880be32f4db1d9f6e2bec38fb6ac23ab,44.90,7.16
88,0036757472ece3dde52fd4bfd929c90e,2,2b939dc9b176d7fa21594d588815d4a4,102.90,14.45
92,0039500db36d50d7d0bfaf23723550f3,2,13944d17b257432717fd260e69853140,59.90,9.94


In [ ]:

# Find products with missing weight or dimensions
display(
    products_work[
        products_work[
            [
                "weight_from_product_g",
                "product_length_cm",
                "product_height_cm",
                "product_width_cm"
            ]
        ].isna().any(axis=1)
    ]
)

# Inspect unmatched product records
display(
    merged.loc[
        merged["product_match_status"] == "Unmatched",
        [
            "product_id",
            "price",
            "freight_value",
            "product_category_name"
        ]
    ].head(10)
)

,product_id,category_from_product,product_name_length,product_description_length,product_photos_qty,weight_from_product_g,product_length_cm,product_height_cm,product_width_cm
8412,09ff539a621711667c43eba6a3bd8466,bebes,60,865,3,NaN,NaN,NaN,NaN


,product_id,price,freight_value,product_category_name
123,ff6caf9340512b8bf6d2a2a6df032cfa,7.79,7.78,NaN
125,a9c404971d1a5b1cbc2e4070e02731fd,7.60,10.96,NaN
132,5a848e4ab52fd5445cdc07aab1c40e48,122.99,15.61,NaN
142,41eee23c25f7a574dfaf8d5c151dbb12,20.30,16.79,NaN
171,e10758160da97891c2fdcbc35f0f031d,56.00,14.14,NaN
370,76d1a1a9d21ab677a61c3ae34b1b352f,159.77,16.00,NaN
378,fbb1cfc2810efabf3235eccf4530f4ae,44.90,16.11,NaN
434,5a848e4ab52fd5445cdc07aab1c40e48,122.99,9.06,NaN
454,9f69acd4da62618a3f6365b732d00ccd,33.90,13.82,NaN
456,ea11e700a343582ad56e4c70e966cb36,593.36,98.13,NaN


In [ ]:

from pathlib import Path
import pandas as pd

# Create working copies; preserve the original data
orders_clean = orders_work.copy()
products_clean = products_work.copy()
merged_clean = merged.copy()

# Convert dates
orders_clean["shipping_limit_date"] = pd.to_datetime(
    orders_clean["shipping_limit_date"], errors="coerce"
)

merged_clean["shipping_limit_date"] = pd.to_datetime(
    merged_clean["shipping_limit_date"], errors="coerce"
)

# Standardize category text without changing missing values
orders_clean["category_from_order"] = (
    orders_clean["category_from_order"]
    .astype("string").str.strip().str.lower()
)

products_clean["category_from_product"] = (
    products_clean["category_from_product"]
    .astype("string").str.strip().str.lower()
)

merged_clean["product_category_name"] = (
    merged_clean["product_category_name"]
    .astype("string").str.strip().str.lower()
)

# Ensure numeric columns have numeric data types
order_numeric = [
    "price", "freight_value", "weight_from_order_g"
]

product_numeric = [
    "product_name_length",
    "product_description_length",
    "product_photos_qty",
    "weight_from_product_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

for col in order_numeric:
    orders_clean[col] = pd.to_numeric(
        orders_clean[col], errors="coerce"
    )

for col in product_numeric:
    products_clean[col] = pd.to_numeric(
        products_clean[col], errors="coerce"
    )

print("Standardization completed.")
print("Order items:", orders_clean.shape)
print("Products:", products_clean.shape)
print("Merged dataset:", merged_clean.shape)

Standardization completed.
Order items: (57373, 9)
Products: (32341, 9)
Merged dataset: (57373, 16)


In [ ]:

print("ROWS BEFORE CLEANING:", len(orders))
print("ROWS AFTER CLEANING:", len(merged_clean))

print(
    "Duplicate order-item keys:",
    merged_clean.duplicated(
        subset=["order_id", "order_item_id"]
    ).sum()
)

print(
    "Unique order IDs:",
    merged_clean["order_id"].nunique()
)

print(
    "Orders with multiple rows:",
    (merged_clean.groupby("order_id").size() > 1).sum()
)

print(
    "Missing categories:",
    merged_clean["product_category_name"].isna().sum()
)

print(
    "Missing product weights:",
    merged_clean["product_weight_g"].isna().sum()
)

print(
    "Invalid shipping dates:",
    merged_clean["shipping_limit_date"].isna().sum()
)

print(
    "Non-positive prices:",
    (merged_clean["price"] <= 0).sum()
)

print(
    "Zero freight values:",
    (merged_clean["freight_value"] == 0).sum()
)

ROWS BEFORE CLEANING: 57373
ROWS AFTER CLEANING: 57373
Duplicate order-item keys: 0
Unique order IDs: 57373
Orders with multiple rows: 0
Missing categories: 820
Missing product weights: 820
Invalid shipping dates: 0
Non-positive prices: 0
Zero freight values: 188


In [ ]:

# Create the output folder
output_path = Path("../data/cleaned")
output_path.mkdir(parents=True, exist_ok=True)

# Save cleaned tables
orders_clean.to_csv(
    output_path / "order_items_clean.csv",
    index=False
)

products_clean.to_csv(
    output_path / "products_clean.csv",
    index=False
)

merged_clean.to_csv(
    output_path / "ecommerce_merged_clean.csv",
    index=False
)

print("Three cleaned CSV files saved successfully.")

Three cleaned CSV files saved successfully.


In [ ]:
from pathlib import Path

print("Current working directory:", Path.cwd())
print("Cleaned data folder:", output_path.resolve())

print("\nFiles saved:")
for file in output_path.glob("*.csv"):
    print(file.name, "-", round(file.stat().st_size / 1024, 2), "KB")

Current working directory: /content
Cleaned data folder: /data/cleaned

Files saved:
ecommerce_merged_clean.csv - 10762.38 KB
products_clean.csv - 2506.32 KB
order_items_clean.csv - 7847.06 KB


In [ ]:
import shutil
from pathlib import Path

cleaned_folder = output_path.resolve()

zip_path = shutil.make_archive(
    str(cleaned_folder.parent / "ecommerce_cleaned_data"),
    "zip",
    root_dir=cleaned_folder
)

print("ZIP file created:", zip_path)

ZIP file created: /data/ecommerce_cleaned_data.zip


In [ ]:
from google.colab import files

files.download(zip_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>